In [0]:
%pip install great-expectations==0.17.23
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.6/813.6 kB 39.3 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Import Libraries
import great_expectations as ge
from pyspark.sql.functions import col, monotonically_increasing_id, count
from pyspark.sql.window import Window


In [0]:
airlines_bronze_path = "s3://travel-analytics-bronze/delta/bronze/airlines/"
df_airlines = spark.read.format("delta").load(airlines_bronze_path)

print("=" * 80)
print("AIRLINES VALIDATION WITH GREAT EXPECTATIONS")
print("=" * 80)
print(f"Total records: {df_airlines.count()}")
print("\n--- Schema ---")
df_airlines.printSchema()

AIRLINES VALIDATION WITH GREAT EXPECTATIONS
Total records: 1251

--- Schema ---
root
 |-- _airbyte_ab_id: string (nullable = true)
 |-- _airbyte_emitted_at: timestamp (nullable = true)
 |-- alias: string (nullable = true)
 |-- country: string (nullable = true)
 |-- airline_id: long (nullable = true)
 |-- fleet_size: long (nullable = true)
 |-- _ab_cdc_lsn: double (nullable = true)
 |-- airline_iata: string (nullable = true)
 |-- airline_icao: string (nullable = true)
 |-- airline_name: string (nullable = true)
 |-- _ab_cdc_deleted_at: string (nullable = true)
 |-- _ab_cdc_updated_at: string (nullable = true)
 |-- _airbyte_additional_properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



In [0]:
# Create Great Expectations DataFrame
ge_df = ge.from_pandas(df_airlines.toPandas())

print("\nRunning Great Expectations validation...")



Running Great Expectations validation...


In [0]:
# Define and Run Expectations

# Expectation 1: airline_id NOT NULL
result1 = ge_df.expect_column_values_to_not_be_null(column="airline_id")
print(f"✓ airline_id NOT NULL: {result1['success']}")

# Expectation 2: airline_id UNIQUE
result2 = ge_df.expect_column_values_to_be_unique(column="airline_id")
print(f"✓ airline_id UNIQUE: {result2['success']}")

# Expectation 3: country NOT NULL
result3 = ge_df.expect_column_values_to_not_be_null(column="country")
print(f"✓ country NOT NULL: {result3['success']}")

✓ airline_id NOT NULL: True
✓ airline_id UNIQUE: True
✓ country NOT NULL: True


In [0]:
# STEP 5: Extract Failed Rows from Great Expectations Results

failed_rows = set()

# Add row_id for tracking
df_with_id = df_airlines.withColumn("row_id", monotonically_increasing_id())

# Extract failed rows for airline_id NULL
if not result1['success']:
    null_rows = df_with_id.filter(col("airline_id").isNull())
    failed_rows.update([row['row_id'] for row in null_rows.collect()])
    print(f"  → Found {null_rows.count()} rows with airline_id = NULL")

# Extract failed rows for airline_id NOT UNIQUE
if not result2['success']:
    window_spec = Window.partitionBy("airline_id")
    df_temp = df_with_id.withColumn("count", count("airline_id").over(window_spec))
    duplicate_rows = df_temp.filter(col("count") > 1)
    failed_rows.update([row['row_id'] for row in duplicate_rows.collect()])
    print(f"  → Found {duplicate_rows.count()} rows with duplicate airline_id")

# Extract failed rows for country NULL
if not result3['success']:
    null_country_rows = df_with_id.filter(col("country").isNull())
    failed_rows.update([row['row_id'] for row in null_country_rows.collect()])
    print(f"  → Found {null_country_rows.count()} rows with country = NULL")
    

In [0]:
# Split Valid and Invalid Records

if len(failed_rows) > 0:
    df_invalid = df_with_id.filter(col("row_id").isin(list(failed_rows))).drop("row_id")
    df_valid = df_with_id.filter(~col("row_id").isin(list(failed_rows))).drop("row_id")
else:
    df_valid = df_airlines
    df_invalid = spark.createDataFrame([], df_airlines.schema)

valid_count = df_valid.count()
invalid_count = df_invalid.count()
total_count = df_airlines.count()

print("\n" + "=" * 80)
print("VALIDATION RESULTS")
print("=" * 80)
print(f" Valid records:   {valid_count} ({valid_count/total_count*100:.2f}%)")
print(f" Invalid records: {invalid_count} ({invalid_count/total_count*100:.2f}%)")



VALIDATION RESULTS
 Valid records:   1251 (100.00%)
 Invalid records: 0 (0.00%)


In [0]:
# Write Invalid Records to Quarantine

if invalid_count > 0:
    quarantine_path = "s3://travel-analytics-bronze/Quarantine/Airlines"
    
    df_invalid.write \
        .format("parquet") \
        .mode("append") \
        .save(quarantine_path)
    
    print(f"\n❌ Invalid records sent to Quarantine: {quarantine_path}")
    print("\n--- Sample Invalid Records ---")
    df_invalid.show(10, truncate=False)
else:
    print("\n✅ All records passed validation! No quarantine needed.")

print("\n" + "=" * 80)
print("✅ VALIDATION COMPLETED!")
print("=" * 80)
print(f"Valid records remain in Bronze: {airlines_bronze_path}")
print(f"Invalid records in Quarantine: s3://travel-analytics-bronze/Quarantine/Airlines")


✅ All records passed validation! No quarantine needed.

✅ VALIDATION COMPLETED!
Valid records remain in Bronze: s3://travel-analytics-bronze/delta/bronze/airlines/
Invalid records in Quarantine: s3://travel-analytics-bronze/Quarantine/Airlines
